# LoCED 100M body — 12 layers, dynamic mHC + MagicNorm, GPT-2 vocab

Trains a **12-layer LoCED** model with **dynamic mHC** and **MagicNorm** on
synthetic GPT-2-token data.

| Piece | Params |
|---|---:|
| 12-layer transformer body | **99.75M** |
| GPT-2 embedding | 12.87M |
| GPT-2 output head | 12.87M |
| **Full model** | **125.48M** |
| **Active body per token** | **5.30M** |

Per your choice, the 100M target is the **transformer body**; the GPT-2
embedding and head sit on top. “Active body” counts attention, norms, mHC,
the router, the shared expert, and the top-1 routed expert for one token,
across all 12 physical layers. It excludes the embedding lookup and the
vocabulary head (including the head gives 18.17M active). The body is
**5.32% active**.

**Model**: `d=256`, 12 physical layers, 24 visits (`12 × full` then
`6 × reuse + 6 × reindex`), 4 mHC streams with dynamic coefficients,
MagicNorm at every visit exit, 62 routed experts (top-1) + 1 shared expert,
`I=168`, GPT-2 vocabulary (`50,257`).

**Schedule**: `reuse_p` stays 0 for the first 20% of steps (exact FULL
attention as the oracle), ramps to 1 over the next 30%, then stays at 1 so the
sparse indexer-selection path trains.

This is a correctness/plumbing run on synthetic data — not a quality or
throughput claim.

## 1. Environment

The notebook runs the **pure-JAX reference path**, not MaxText. It installs
the MaxText accelerator extra into an isolated Python 3.12 venv so that JAX
comes with the right TPU/CUDA build, then executes the training script inside
that venv.

- **Kaggle TPU**: accelerator → *TPU v5e-8* (8 chips).
- **Kaggle GPU**: accelerator → *GPU T4 ×2*.
- **Colab TPU**: runtime → *TPU* (v5e-1, 1 chip).

The script is written to disk, run with the venv interpreter, and its output
is streamed back here. It ends with a machine-readable JSON report.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

ACCELERATOR = os.environ.get("LOCED_ACCELERATOR", "TPU")
MAXTEXT_REF = "maxtext-v0.2.5"
MAXTEXT_REPO = "https://github.com/AI-Hypercomputer/maxtext.git"
MAXTEXT_EXTRAS = "TPU" if ACCELERATOR == "TPU" else "cuda12"

LOCED_REPO_URL = os.environ.get(
    "LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git"
)
LOCED_REPO_REF = os.environ.get("LOCED_REPO_REF", "master")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
MAXTEXT_DIR = WORK / "maxtext-v0.2.5"
VENV_DIR = WORK / "loced-venv"
PY = VENV_DIR / "bin" / "python"
SCRIPTS = WORK / "loced_scripts"
CACHE = WORK / "jax_cache"
for _p in (SCRIPTS, CACHE):
    _p.mkdir(parents=True, exist_ok=True)


def _loced_parent(root: Path):
    # offx-zinth/loced is the package contents at the repo root.
    if (root / "maxtext" / "train.py").is_file():
        return root.parent
    return None


def _find_loced_root():
    for root in (WORK, WORK / "loced", Path.cwd(), Path.cwd() / "loced"):
        if (root / "loced" / "maxtext" / "train.py").is_file():
            return root
        if (root / "maxtext" / "train.py").is_file():
            return root.parent
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for p in kaggle_input.glob("**/loced/maxtext/train.py"):
            return p.parents[2]
    return None


LOCED_ROOT = _find_loced_root()
if LOCED_ROOT is None:
    target = WORK / "loced"
    if target.exists():
        shutil.rmtree(target)
    print(f"cloning {LOCED_REPO_URL} ({LOCED_REPO_REF}) -> {target}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", LOCED_REPO_REF,
         LOCED_REPO_URL, str(target)],
        check=True,
    )
    LOCED_ROOT = _find_loced_root()
if LOCED_ROOT is None:
    raise RuntimeError("could not locate the loced package")

print("LOCED_ROOT :", LOCED_ROOT)

# --- isolated venv with the MaxText accelerator extra -----------------------
if shutil.which("uv") is None:
    print("installing uv ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")

if not PY.exists():
    print("creating Python 3.12 venv ...")
    subprocess.run([UV, "venv", "--python", "3.12", "--seed", str(VENV_DIR)], check=True)

if not (MAXTEXT_DIR / "pyproject.toml").is_file():
    print(f"cloning MaxText {MAXTEXT_REF} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", MAXTEXT_REF,
         MAXTEXT_REPO, str(MAXTEXT_DIR)],
        check=True,
    )

print(f"installing maxtext[{MAXTEXT_EXTRAS}] (resolution=lowest) ...")
subprocess.run(
    [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
     "-e", f".[{MAXTEXT_EXTRAS}]", "--resolution=lowest"],
    cwd=str(MAXTEXT_DIR), check=True,
)

BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "PYTHONPATH": f"{MAXTEXT_DIR / 'src'}:{LOCED_ROOT}",
    "JAX_COMPILATION_CACHE_DIR": str(CACHE),
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.90",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "true",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
    "LOCED_ROOT": str(LOCED_ROOT),
    "LOCED_ACCELERATOR": ACCELERATOR.lower(),
    "LOCED_WORK": str(WORK),
})

# jax-cuda12 loads pip-installed NVIDIA libraries with ctypes; in a venv they
# live under site-packages/nvidia/*/lib, which is not on the default loader path.
_nvidia_code = (
    "import site; from pathlib import Path; "
    "print(':'.join(str(p) for r in site.getsitepackages() "
    "for p in sorted((Path(r) / 'nvidia').glob('*/lib')) if p.is_dir()))"
)
_nvidia_libs = subprocess.check_output([str(PY), "-c", _nvidia_code], text=True).strip()
if _nvidia_libs:
    BASE_ENV["LD_LIBRARY_PATH"] = ":".join(
        x for x in (_nvidia_libs, os.environ.get("LD_LIBRARY_PATH", "")) if x
    )
print("NVIDIA lib dirs:", len(_nvidia_libs.split(":")) if _nvidia_libs else 0)

print(subprocess.check_output(
    [str(PY), "-c", "import sys, jax; print(sys.version); print('jax', jax.__version__)"],
    text=True,
))

## 2. Train on Colab TPU v5e-1

Runs the full script on the single Colab TPU chip. Free Colab exposes one
v5e device, so this validates the TPU backend and single-device training;
it does not exercise the 8-device data-parallel mesh (use the Kaggle v5e-8
notebook for that).

The first cell installs MaxText's TPU extra into an isolated Python 3.12
venv, which takes roughly 5–15 minutes. Keep the tab open for the whole run.

In [ ]:
SCRIPT = "\"\"\"LoCED 100M-body / 12-layer / dynamic-mHC+MagicNorm / GPT-2 vocab.\n\nSelf-contained script executed by the Kaggle notebook inside an isolated\nPython 3.12 venv that has the MaxText accelerator extra installed.\n\nStages\n  1. build the 12-layer config and audit the parameter budget\n  2. install a sparse top-1 MoE dispatch (ragged_dot) with a dense fallback\n  3. verify the data-parallel gradient reduction on the real accelerator\n  4. train on synthetic GPT-2-token data\n  5. evaluate, generate, and write a machine-readable report\n\"\"\"\n\nimport dataclasses\nimport functools\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nimport jax\nimport jax.numpy as jnp\nimport optax\n\nfrom loced.block import sinkhorn\nfrom loced.config import LoCEDConfig\nfrom loced.model import count_params, forward_train, generate, init_model\nfrom loced.train import make_optimizer, tiny_config\n\nACCEL = os.environ.get(\"LOCED_ACCELERATOR\", \"tpu\")\nWORK = Path(os.environ.get(\"LOCED_WORK\", \"/kaggle/working\"))\nWORK.mkdir(parents=True, exist_ok=True)\n\nSEQ_LEN = int(os.environ.get(\"LOCED_SEQ_LEN\", \"64\"))\nDECODE_LEN = int(os.environ.get(\"LOCED_DECODE_LEN\", \"16\"))\nN_EXPERTS = int(os.environ.get(\"LOCED_EXPERTS\", \"62\"))\nTOP_K_EXPERTS = 1\nINTERMEDIATE = int(os.environ.get(\"LOCED_INTERMEDIATE\", \"168\"))\nACTIVE_VOCAB = int(os.environ.get(\"LOCED_ACTIVE_VOCAB\", \"512\"))\nRULE_P = float(os.environ.get(\"LOCED_RULE_P\", \"0.9\"))\nLR = float(os.environ.get(\"LOCED_LR\", \"1e-3\"))\nVOCAB_SIZE = int(os.environ.get(\"LOCED_VOCAB\", \"50257\"))\n\nN_DEVICES = len(jax.devices())\nDEFAULT_BATCH = 8 if N_DEVICES >= 8 else max(1, 2 * N_DEVICES)\nGLOBAL_BATCH = int(os.environ.get(\"LOCED_BATCH\", str(DEFAULT_BATCH)))\nGLOBAL_BATCH = max(N_DEVICES, (GLOBAL_BATCH // N_DEVICES) * N_DEVICES)\nPER_DEVICE_BATCH = GLOBAL_BATCH // N_DEVICES\nDEFAULT_STEPS = 20 if N_DEVICES >= 2 else 3\nSTEPS = int(os.environ.get(\"LOCED_STEPS\", str(DEFAULT_STEPS)))\nMAX_NEW = int(os.environ.get(\"LOCED_MAX_NEW\", \"16\"))\nRUN_GENERATION = os.environ.get(\"LOCED_RUN_GENERATION\", \"1\") == \"1\"\n\nprint(\"=\" * 70)\nprint(f\"accelerator : {ACCEL}  backend: {jax.default_backend()}  \"\n      f\"devices: {N_DEVICES}\")\nprint(f\"jax         : {jax.__version__}   optax: {optax.__version__}\")\nprint(f\"batch       : global {GLOBAL_BATCH} / per-device {PER_DEVICE_BATCH}\")\nprint(f\"seq_len     : {SEQ_LEN}   steps: {STEPS}   lr: {LR}\")\nprint(\"=\" * 70, flush=True)\n\nEXPECTED_BACKEND = os.environ.get(\"LOCED_EXPECTED_BACKEND\", \"\")\nEXPECTED_DEVICES = int(os.environ.get(\"LOCED_EXPECTED_DEVICES\", \"0\"))\nMIN_DEVICES = int(os.environ.get(\"LOCED_MIN_DEVICES\", \"0\"))\nif EXPECTED_BACKEND and jax.default_backend() != EXPECTED_BACKEND:\n    raise RuntimeError(\n        f\"expected backend {EXPECTED_BACKEND!r}, got \"\n        f\"{jax.default_backend()!r}; pick the right accelerator and rerun\")\nif EXPECTED_DEVICES and N_DEVICES != EXPECTED_DEVICES:\n    raise RuntimeError(\n        f\"expected {EXPECTED_DEVICES} device(s), got {N_DEVICES}; \"\n        f\"pick the right accelerator and rerun\")\nif MIN_DEVICES and N_DEVICES < MIN_DEVICES:\n    raise RuntimeError(\n        f\"expected at least {MIN_DEVICES} device(s), got {N_DEVICES}\")\n\n\n# ---------------------------------------------------------------------------\n# 1. config + parameter budget\n# ---------------------------------------------------------------------------\n\ndef build_config(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, decode_len=DECODE_LEN,\n                 num_experts=N_EXPERTS):\n    n_layers = 12\n    encode = tuple(\"full\" for _ in range(n_layers))\n    decode = encode + tuple(\"reuse\" for _ in range(6)) + tuple(\"reindex\" for _ in range(6))\n    return LoCEDConfig(\n        vocab_size=vocab_size,\n        hidden_size=256,\n        max_seq_len=seq_len,\n        max_decode_len=decode_len,\n        magic_norm=True,\n        mhc_streams=4,\n        mhc_sinkhorn_iters=10,\n        mhc_dynamic=True,\n        num_attention_heads=8,\n        head_dim=32,\n        kv_dim=32,\n        pool_m=2,\n        window=min(32, seq_len // 2),\n        top_k=min(32, seq_len // 2),\n        indexer_heads=4,\n        indexer_dim=64,\n        indexer_block=4,\n        hier_blocks=4,\n        encode_layers=encode,\n        decode_layers=decode,\n        num_experts=num_experts,\n        num_shared_experts=1,\n        top_k_experts=TOP_K_EXPERTS,\n        intermediate_size=INTERMEDIATE,\n    )\n\n\ndef count_active_body(cfg, params):\n    \"\"\"Body weights one token touches: attention, norms, mHC, router, the\n    shared expert, and exactly top_k routed experts per physical layer.\"\"\"\n    n = 0\n    for name in (\"W_bk\", \"W_bv\", \"W_ik\", \"W_iq\"):\n        n += int(params[name].size)\n    for name in (\"cond\", \"gate_a\", \"gate_m\", \"norm_f\",\n                 \"mhc_res\", \"mhc_pre\", \"mhc_post\", \"mhc_dyn\"):\n        n += int(params[name].size)\n    for cell in params[\"cells\"]:\n        for name in (\"norm_a\", \"norm_m\", \"norm_magic\", \"W_q\", \"W_fx\", \"W_fv\",\n                     \"W_o\", \"W_gate\", \"Ws1\", \"Ws3\", \"Ws2\"):\n            n += int(cell[name].size)\n        per_expert = int(cell[\"We1\"][0].size + cell[\"We3\"][0].size\n                         + cell[\"We2\"][0].size)\n        n += cfg.top_k_experts * per_expert\n    return int(n)\n\n\ncfg = build_config()\nprint(f\"layers={cfg.num_layers} visits={cfg.num_visits} \"\n      f\"(encode {cfg.prefill_visits} + decode {cfg.num_visits - cfg.prefill_visits})\")\nprint(f\"sparse pass : {cfg.sparse_layers}\")\nprint(f\"mHC         : streams={cfg.mhc_streams} dynamic={cfg.mhc_dynamic} \"\n      f\"sinkhorn_iters={cfg.mhc_sinkhorn_iters} magic_norm={cfg.magic_norm}\")\nprint(f\"MoE         : {cfg.num_experts} routed, top-{cfg.top_k_experts}, \"\n      f\"{cfg.num_shared_experts} shared, I={cfg.intermediate_size}\", flush=True)\n\nparams = init_model(cfg, jax.random.PRNGKey(0))\ntotal = count_params(params)\nembed = int(params[\"embed\"].size)\nhead = int(params[\"head\"].size)\nbody = total - embed - head\nactive_body = count_active_body(cfg, params)\nactive_with_head = active_body + head + cfg.hidden_size\n\nmhc = np.asarray(sinkhorn(params[\"mhc_res\"], cfg.mhc_sinkhorn_iters))\nrow_err = float(np.max(np.abs(mhc.sum(axis=-1) - 1.0)))\ncol_err = float(np.max(np.abs(mhc.sum(axis=-2) - 1.0)))\n\nprint(f\"total parameters          : {total:>14,}  ({total / 1e6:8.3f} M)\")\nprint(f\"  transformer body        : {body:>14,}  ({body / 1e6:8.3f} M)\")\nprint(f\"  GPT-2 embedding         : {embed:>14,}  ({embed / 1e6:8.3f} M)\")\nprint(f\"  GPT-2 output head       : {head:>14,}  ({head / 1e6:8.3f} M)\")\nprint(f\"active body per token     : {active_body:>14,}  ({active_body / 1e6:8.3f} M)\")\nprint(f\"active incl. head + row   : {active_with_head:>14,}  ({active_with_head / 1e6:8.3f} M)\")\nprint(f\"body sparsity             : {active_body / body:.2%} active\")\nprint(f\"Sinkhorn max row/col err  : {max(row_err, col_err):.3e}\", flush=True)\n\nassert 95e6 <= body <= 105e6, f\"body budget off: {body}\"\nassert 5e6 <= active_body <= 6e6, f\"active budget off: {active_body}\"\nassert max(row_err, col_err) < 1e-4, \"Sinkhorn projection is not doubly stochastic\"\n\n\n# ---------------------------------------------------------------------------\n# 2. sparse top-1 MoE dispatch (notebook-local, repo untouched)\n# ---------------------------------------------------------------------------\n\nimport loced.block as loced_block\nfrom loced.fnn import _swiglu, init_moe\n\nDENSE_MOE_FORWARD = loced_block.moe_forward\n\n\ndef _ragged_dot_supported():\n    try:\n        lhs = jnp.zeros((4, 8), jnp.float32)\n        rhs = jnp.zeros((2, 8, 4), jnp.float32)\n        sizes = jnp.array([2, 2], jnp.int32)\n        out = jax.jit(lambda a, b, s: jax.lax.ragged_dot(a, b, s))(\n            lhs, rhs, sizes)\n        out.block_until_ready()\n        return True\n    except Exception as exc:  # pragma: no cover - backend dependent\n        print(f\"ragged_dot unavailable on this backend ({exc}); \"\n              f\"falling back to the dense reference MoE\")\n        return False\n\n\ndef _psum_div(x, axis=\"data\"):\n    \"\"\"Average a batch statistic over the data-parallel axis.\n\n    Inside shard_map the MoE load-balance auxiliary loss must use GLOBAL\n    batch statistics: it is a product of two batch-level means, so a per-shard\n    value cannot be recombined into the global one. Outside shard_map the\n    axis is unbound and this is the identity.\n    \"\"\"\n    try:\n        return jax.lax.psum(x, axis) / N_DEVICES\n    except Exception as exc:  # unbound 'data' axis outside shard_map\n        if \"axis\" not in str(exc).lower():\n            raise\n        return x\n\n\ndef _global_aux_moe(cfg, p, x):\n    \"\"\"Dense reference MoE whose load-balance auxiliary uses global stats.\"\"\"\n    shared = _swiglu(x @ p[\"Ws1\"], x @ p[\"Ws3\"]) @ p[\"Ws2\"]\n    a = jnp.einsum(\"btd,edi->btei\", x, p[\"We1\"])\n    g = jnp.einsum(\"btd,edi->btei\", x, p[\"We3\"])\n    h = _swiglu(a, g)\n    y = jnp.einsum(\"btei,eid->bted\", h, p[\"We2\"])\n    probs = jax.nn.softmax(x @ p[\"W_gate\"], axis=-1)\n    k = cfg.top_k_experts\n    top_idx = jax.lax.top_k(probs, k)[1]\n    w = jnp.take_along_axis(probs, top_idx, axis=-1)\n    w = w / jnp.maximum(w.sum(-1, keepdims=True), 1e-9)\n    mask = jax.nn.one_hot(top_idx, cfg.num_experts, dtype=jnp.float32)\n    mask = (mask * w[..., None]).sum(axis=2)\n    routed = jnp.einsum(\"bte,bteh->bth\", mask, y)\n    out = shared + routed\n    frac = jnp.mean((mask > 0).astype(jnp.float32), axis=(0, 1))\n    mean_p = jnp.mean(probs, axis=(0, 1))\n    aux = cfg.num_experts * jnp.sum(_psum_div(frac) * _psum_div(mean_p))\n    return out, aux\n\n\ndef sparse_top1_moe(cfg, p, x):\n    \"\"\"Top-1 MoE: a token reads only its selected expert's three SwiGLU\n    matrices. Mathematically identical to the dense kernel for top-1.\"\"\"\n    B, T, d = x.shape\n    shared = _swiglu(x @ p[\"Ws1\"], x @ p[\"Ws3\"]) @ p[\"Ws2\"]\n\n    xf = x.reshape(B * T, d)\n    probs = jax.nn.softmax(xf @ p[\"W_gate\"], axis=-1)\n    top_w, top_idx = jax.lax.top_k(probs, 1)\n    top_w = top_w / jnp.maximum(top_w.sum(-1, keepdims=True), 1e-9)\n    flat_idx = top_idx[:, 0]\n\n    perm = jnp.argsort(flat_idx)\n    inv = jnp.argsort(perm)\n    xs = xf[perm]\n    sizes = jnp.bincount(flat_idx, length=cfg.num_experts)\n\n    a = jax.lax.ragged_dot(xs, p[\"We1\"], sizes)\n    g = jax.lax.ragged_dot(xs, p[\"We3\"], sizes)\n    h = _swiglu(a, g)\n    y = jax.lax.ragged_dot(h, p[\"We2\"], sizes)\n    routed = (y * top_w[:, 0][:, None])[inv].reshape(B, T, d)\n\n    frac = jnp.mean(\n        jax.nn.one_hot(flat_idx, cfg.num_experts, dtype=jnp.float32), axis=0)\n    mean_p = jnp.mean(probs, axis=0)\n    aux = cfg.num_experts * jnp.sum(_psum_div(frac) * _psum_div(mean_p))\n    return shared + routed, aux\n\n\nUSE_SPARSE = False\nif TOP_K_EXPERTS == 1 and _ragged_dot_supported():\n    tiny = dataclasses.replace(tiny_config(), top_k_experts=1)\n    tiny_params = init_moe(tiny, jax.random.PRNGKey(1))\n    tiny_x = jax.random.normal(jax.random.PRNGKey(2), (2, 8, tiny.hidden_size))\n    dense_out, dense_aux = _global_aux_moe(tiny, tiny_params, tiny_x)\n    sparse_out, sparse_aux = sparse_top1_moe(tiny, tiny_params, tiny_x)\n    value_err = float(jnp.max(jnp.abs(dense_out - sparse_out)))\n    aux_err = float(jnp.abs(dense_aux - sparse_aux))\n    dense_grad = jax.grad(\n        lambda q: _global_aux_moe(tiny, q, tiny_x)[0].sum())(tiny_params)\n    sparse_grad = jax.grad(\n        lambda q: sparse_top1_moe(tiny, q, tiny_x)[0].sum())(tiny_params)\n    grad_err = max(\n        float(jnp.max(jnp.abs(a - b)))\n        for a, b in zip(jax.tree.leaves(dense_grad), jax.tree.leaves(sparse_grad)))\n    print(f\"top-1 dense-vs-sparse parity: value={value_err:.3e} \"\n          f\"aux={aux_err:.3e} grad={grad_err:.3e}\", flush=True)\n    if value_err < 1e-6 and aux_err < 1e-6 and grad_err < 1e-5:\n        loced_block.moe_forward = sparse_top1_moe\n        USE_SPARSE = True\n        print(\"installed sparse top-1 MoE dispatch (ragged_dot)\")\n    else:\n        print(\"sparse dispatch failed parity; keeping the dense reference MoE\")\nelse:\n    print(\"using the dense reference MoE (top_k_experts != 1 or no ragged_dot)\")\n\n\n# ---------------------------------------------------------------------------\n# 3. data-parallel gradient reduction on the real accelerator\n# ---------------------------------------------------------------------------\n\ndef _shard_map(fn, mesh, in_specs, out_specs):\n    \"\"\"Version-tolerant shard_map (jax.shard_map vs the experimental one).\"\"\"\n    try:\n        return jax.shard_map(fn, mesh=mesh, in_specs=in_specs, out_specs=out_specs)\n    except AttributeError:  # pragma: no cover - older jax\n        from jax.experimental.shard_map import shard_map\n        return shard_map(fn, mesh, in_specs, out_specs)\n\ndef mesh_grad_check():\n    \"\"\"Compare the sharded gradient against the single-device gradient.\n\n    Inside shard_map the gradient of a REPLICATED parameter is already\n    all-reduced (summed) across the data axis, so the local loss must be\n    scaled by 1/N_DEVICES for the result to match the single-device gradient\n    of the global mean. This check validates that on the actual accelerator.\n    \"\"\"\n    if N_DEVICES == 1:\n        return {\"checked\": False, \"reason\": \"single device\"}\n\n    from jax.sharding import Mesh, NamedSharding, PartitionSpec as P\n\n    tiny = dataclasses.replace(tiny_config(), top_k_experts=1)\n    tokens = np.zeros((N_DEVICES, tiny.max_seq_len), dtype=np.int32)\n    rng = np.random.default_rng(0)\n    for i in range(N_DEVICES):\n        row = rng.integers(0, tiny.vocab_size, size=tiny.max_seq_len)\n        row[0] = i % tiny.vocab_size\n        tokens[i] = row\n    tokens = jnp.asarray(tokens)\n\n    small_params = init_model(tiny, jax.random.PRNGKey(3))\n\n    def loss_of(p, toks):\n        return forward_train(tiny, p, toks, reuse_p=0.0)[\"loss\"]\n\n    g_single = jax.grad(lambda p: loss_of(p, tokens))(small_params)\n    mesh = Mesh(np.array(jax.devices()), axis_names=(\"data\",))\n    replicated = NamedSharding(mesh, P())\n    batch_sharded = NamedSharding(mesh, P(\"data\"))\n\n    def sharded_loss(p, toks):\n        # toks is the LOCAL shard of the batch; the loss is a mean over the\n        # local rows, so dividing by N_DEVICES turns the cross-shard sum into\n        # the global mean.\n        return loss_of(p, toks) / N_DEVICES\n\n    sharded_grad_fn = _shard_map(\n        lambda p, toks: jax.grad(sharded_loss)(p, toks),\n        mesh=mesh,\n        in_specs=(P(), P(\"data\")),\n        out_specs=P(),\n    )\n    sharded_grad_fn = jax.jit(sharded_grad_fn)\n\n    p_dev = jax.device_put(small_params, replicated)\n    t_dev = jax.device_put(tokens, batch_sharded)\n    g_sharded = sharded_grad_fn(p_dev, t_dev)\n\n    diffs = [\n        float(jnp.max(jnp.abs(a - b)))\n        for a, b in zip(jax.tree.leaves(g_single), jax.tree.leaves(g_sharded))\n    ]\n    max_diff = max(diffs)\n    scale = max(1e-6, max(float(jnp.max(jnp.abs(a)))\n                          for a in jax.tree.leaves(g_single)))\n    rel = max_diff / scale\n    print(f\"mesh gradient check: max abs diff {max_diff:.3e}  \"\n          f\"relative {rel:.3e}  ({N_DEVICES} devices)\", flush=True)\n    if rel > 1e-2:\n        raise RuntimeError(\n            f\"data-parallel gradient mismatch (relative {rel:.3e}); the \"\n            f\"sharded gradient does not match the single-device gradient\")\n    if rel > 1e-4:\n        print(\"WARNING: mesh gradient relative difference above 1e-4 \"\n              \"(float32 accumulation noise); continuing\", flush=True)\n    return {\"checked\": True, \"max_abs_diff\": max_diff, \"relative_diff\": rel,\n            \"within_1e-4\": bool(rel <= 1e-4), \"devices\": N_DEVICES}\n\n\nGRAD_CHECK = mesh_grad_check()\n\n\n# ---------------------------------------------------------------------------\n# 4. synthetic GPT-2-token data + training\n# ---------------------------------------------------------------------------\n\ndef synthetic_batch_gpt2(rng, batch, seq, rule_p=RULE_P):\n    \"\"\"Causal local rule: next token is (prev + 1) % ACTIVE_VOCAB with\n    probability rule_p, else uniform noise over the same subset of GPT-2\n    token IDs. The model keeps the full 50,257-way GPT-2 head.\"\"\"\n    tok = np.zeros((batch, seq), dtype=np.int32)\n    tok[:, 0] = rng.integers(0, ACTIVE_VOCAB, size=batch)\n    for t in range(1, seq):\n        step = (tok[:, t - 1] + 1) % ACTIVE_VOCAB\n        noise = rng.integers(0, ACTIVE_VOCAB, size=batch)\n        take_rule = rng.random(batch) < rule_p\n        tok[:, t] = np.where(take_rule, step, noise)\n    return jnp.asarray(tok)\n\n\noptimizer = make_optimizer(cfg, params, optim=\"muon\", lr=LR)\nopt_state = optimizer.init(params)\n\nfrom jax.sharding import Mesh, NamedSharding, PartitionSpec as P\n\nmesh = Mesh(np.array(jax.devices()), axis_names=(\"data\",))\nreplicated = NamedSharding(mesh, P())\nbatch_sharded = NamedSharding(mesh, P(\"data\"))\n\nparams = jax.device_put(params, replicated)\nopt_state = jax.device_put(opt_state, replicated)\n\n\ndef _train_body(params, state, tokens, reuse_p):\n    def loss_fn(p):\n        out = forward_train(cfg, p, tokens, reuse_p=reuse_p)\n        metrics = {k: out[k] for k in (\"loss\", \"ce\", \"ce_aux\", \"kl\", \"moe_aux\")}\n        metrics[\"reuse_p\"] = jnp.asarray(reuse_p, jnp.float32)\n        # Scale by 1/N_DEVICES: shard_map already all-reduces the gradient of\n        # a replicated parameter across the data axis, so the local mean loss\n        # must be divided by the device count to reproduce the global mean.\n        return out[\"loss\"] / N_DEVICES, metrics\n\n    (_, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)\n    metrics = jax.lax.pmean(metrics, \"data\")\n    updates, state = optimizer.update(grads, state, params)\n    params = optax.apply_updates(params, updates)\n    return params, state, metrics\n\n\ntrain_step = _shard_map(\n    _train_body,\n    mesh=mesh,\n    in_specs=(P(), P(), P(\"data\"), P()),\n    out_specs=(P(), P(), P()),\n)\ntrain_step = jax.jit(train_step)\n\nwarmup = int(STEPS * cfg.reuse_warmup_frac) if cfg.reuse_onpolicy else STEPS\nramp = max(1, int(STEPS * cfg.reuse_ramp_frac))\nhistory = []\ntrain_rng = np.random.default_rng(1)\n\nprint(f\"\\ntraining: {STEPS} steps, batch {GLOBAL_BATCH} x {SEQ_LEN} tokens, \"\n      f\"reuse ramp from step {warmup} over {ramp} steps\\n\", flush=True)\n\nfor i in range(STEPS):\n    reuse_p = (0.0 if not cfg.reuse_onpolicy\n               else min(max((i - warmup) / ramp, 0.0), 1.0))\n    tokens = synthetic_batch_gpt2(train_rng, GLOBAL_BATCH, SEQ_LEN)\n    tokens = jax.device_put(tokens, batch_sharded)\n    reuse = jax.device_put(jnp.asarray(reuse_p, jnp.float32), replicated)\n\n    t0 = time.perf_counter()\n    params, opt_state, metrics = train_step(params, opt_state, tokens, reuse)\n    row = {k: float(v) for k, v in metrics.items()}\n    row[\"ms_step\"] = (time.perf_counter() - t0) * 1000.0\n    row[\"step\"] = i\n    history.append(row)\n\n    if i == 0 or (i + 1) % max(1, STEPS // 10) == 0 or i == STEPS - 1:\n        print(f\"step {i:3d}  loss {row['loss']:.4f}  ce {row['ce']:.4f}  \"\n              f\"kl {row['kl']:.4f}  moe_aux {row['moe_aux']:.4f}  \"\n              f\"reuse_p {row['reuse_p']:.2f}  {row['ms_step']:.1f} ms\", flush=True)\n\n\n# ---------------------------------------------------------------------------\n# 5. evaluation + generation + report\n# ---------------------------------------------------------------------------\n\nparams_trained = jax.tree.map(\n    lambda x: jax.device_put(x, jax.devices()[0]), params)\n\n\n@jax.jit\ndef eval_forward(p, tokens, reuse_p):\n    out = forward_train(cfg, p, tokens, reuse_p=reuse_p)\n    pred = jnp.argmax(out[\"logits\"][:, :-1], axis=-1)\n    acc = jnp.mean((pred == tokens[:, 1:]).astype(jnp.float32))\n    return out[\"loss\"], out[\"ce\"], out[\"kl\"], acc\n\n\neval_rng = np.random.default_rng(123)\neval_tokens = synthetic_batch_gpt2(eval_rng, 4, SEQ_LEN)\neval_results = {}\nprint()\nfor reuse_p in (0.0, 0.5, 1.0):\n    loss, ce, kl, acc = eval_forward(params_trained, eval_tokens, reuse_p)\n    eval_results[str(reuse_p)] = {\n        \"loss\": float(loss), \"ce\": float(ce), \"kl\": float(kl),\n        \"next_token_acc\": float(acc),\n    }\n    print(f\"eval reuse_p={reuse_p:.1f}  loss={float(loss):.4f}  \"\n          f\"ce={float(ce):.4f}  kl={float(kl):.4f}  \"\n          f\"next-token acc={float(acc):.3f}\", flush=True)\n\ngeneration = None\nif RUN_GENERATION:\n    prompt = synthetic_batch_gpt2(np.random.default_rng(7), 1, min(16, SEQ_LEN // 2))\n    t0 = time.perf_counter()\n    gen = generate(cfg, params_trained, prompt, MAX_NEW)\n    gen_ids = np.asarray(gen)[0]\n    rule_acc = (float(np.mean(gen_ids[1:] == (gen_ids[:-1] + 1) % ACTIVE_VOCAB))\n                if len(gen_ids) > 1 else 0.0)\n    generation = {\n        \"prompt_ids\": np.asarray(prompt)[0].tolist(),\n        \"generated_ids\": gen_ids.tolist(),\n        \"greedy_rule_accuracy\": rule_acc,\n        \"seconds\": time.perf_counter() - t0,\n    }\n    print(f\"generation: {gen_ids.tolist()}  greedy rule accuracy {rule_acc:.3f}\",\n          flush=True)\n\n\ndef _git_commit(path):\n    try:\n        return subprocess.check_output(\n            [\"git\", \"-C\", str(path), \"rev-parse\", \"--short\", \"HEAD\"],\n            text=True).strip()\n    except Exception:\n        return None\n\n\nreport = {\n    \"kind\": \"loced_100m_body_12l_dynamic_mhc_magicnorm_gpt2_synth\",\n    \"accelerator\": ACCEL,\n    \"backend\": jax.default_backend(),\n    \"devices\": N_DEVICES,\n    \"jax_version\": jax.__version__,\n    \"python_version\": sys.version.split()[0],\n    \"vocab_size\": VOCAB_SIZE,\n    \"active_vocab\": ACTIVE_VOCAB,\n    \"repo_commit\": _git_commit(os.environ.get(\"LOCED_ROOT\", \".\")),\n    \"config\": {\n        \"layers\": cfg.num_layers,\n        \"visits\": cfg.num_visits,\n        \"d_model\": cfg.hidden_size,\n        \"heads\": cfg.num_attention_heads,\n        \"head_dim\": cfg.head_dim,\n        \"kv_dim\": cfg.kv_dim,\n        \"seq_len\": cfg.max_seq_len,\n        \"decode_len\": cfg.max_decode_len,\n        \"magic_norm\": cfg.magic_norm,\n        \"mhc_streams\": cfg.mhc_streams,\n        \"mhc_dynamic\": cfg.mhc_dynamic,\n        \"mhc_sinkhorn_iters\": cfg.mhc_sinkhorn_iters,\n        \"num_experts\": cfg.num_experts,\n        \"top_k_experts\": cfg.top_k_experts,\n        \"shared_experts\": cfg.num_shared_experts,\n        \"intermediate_size\": cfg.intermediate_size,\n        \"sparse_layers\": list(cfg.sparse_layers),\n    },\n    \"params\": {\n        \"total\": int(total),\n        \"body\": int(body),\n        \"embedding\": int(embed),\n        \"head\": int(head),\n        \"active_body_per_token\": int(active_body),\n        \"active_with_head_and_embed_row\": int(active_with_head),\n        \"body_active_fraction\": float(active_body / body),\n        \"sinkhorn_max_error\": float(max(row_err, col_err)),\n    },\n    \"moe_dispatch\": \"sparse_top1_ragged_dot\" if USE_SPARSE else \"dense_reference\",\n    \"mesh_gradient_check\": GRAD_CHECK,\n    \"training\": {\n        \"steps\": STEPS,\n        \"global_batch\": GLOBAL_BATCH,\n        \"per_device_batch\": PER_DEVICE_BATCH,\n        \"seq_len\": SEQ_LEN,\n        \"learning_rate\": LR,\n        \"optimizer\": \"muon\",\n        \"rule_p\": RULE_P,\n        \"history\": history,\n        \"mean_ms_per_step_after_first\": (\n            float(np.mean([r[\"ms_step\"] for r in history[1:]])) if len(history) > 1\n            else float(history[0][\"ms_step\"])),\n    },\n    \"eval\": eval_results,\n    \"generation\": generation,\n    \"notes\": [\n        \"100M target is the transformer body; GPT-2 embedding and head are on top.\",\n        \"active_body excludes the embedding lookup and the vocabulary head.\",\n        \"Synthetic data only; not a quality or throughput claim.\",\n    ],\n}\n\nout_path = WORK / \"loced_100m_body_12l_mhc_gpt2_synth_report.json\"\nout_path.write_text(json.dumps(report, indent=2, default=float))\nprint(f\"\\nwrote {out_path}\", flush=True)\n\ntry:\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(figsize=(7, 3.5))\n    ax.plot([r[\"step\"] for r in history], [r[\"loss\"] for r in history], marker=\"o\")\n    ax.set_xlabel(\"step\")\n    ax.set_ylabel(\"loss\")\n    ax.set_title(f\"LoCED 100M body / 12L / dynamic mHC / {ACCEL}\")\n    ax.grid(alpha=0.3)\n    fig.tight_layout()\n    plot_path = WORK / \"loced_100m_body_loss.png\"\n    fig.savefig(plot_path, dpi=140)\n    plt.close(fig)\n    print(f\"wrote {plot_path}\", flush=True)\nexcept Exception as exc:\n    print(f\"plot skipped: {exc}\", flush=True)\n\nprint(\"SCRIPT COMPLETE\", flush=True)\n"

In [ ]:
TRAIN_SCRIPT = SCRIPTS / "loced_arch_100m.py"
TRAIN_SCRIPT.write_text(SCRIPT)
print(f"wrote {TRAIN_SCRIPT} ({len(SCRIPT):,} chars)")

env = BASE_ENV.copy()
env.update({
    "LOCED_tpu": "tpu",
    "LOCED_1": "1",
    "LOCED_0": "0",
})

t0 = time.time()
proc = subprocess.run(
    [str(PY), str(TRAIN_SCRIPT)],
    cwd=str(LOCED_ROOT),
    env=env,
    text=True,
    capture_output=True,
    timeout=int(os.environ.get("LOCED_TIMEOUT", "7200")),
)
elapsed = time.time() - t0
print(proc.stdout)
if proc.stderr.strip():
    print("[stderr]\n" + proc.stderr[-16000:])
print(f"[train] exit={proc.returncode} elapsed={elapsed:.1f}s")

REPORT = WORK / "loced_100m_body_12l_mhc_gpt2_synth_report.json"
if proc.returncode != 0 or not REPORT.is_file():
    raise RuntimeError(f"training failed (exit {proc.returncode}); see output above")

report = json.loads(REPORT.read_text())
print("\n================ REPORT ================")
print("backend           :", report["backend"], "devices:", report["devices"])
print("moe dispatch      :", report["moe_dispatch"])
print("mesh grad check   :", report["mesh_gradient_check"])
p = report["params"]
print(f"total             : {p['total']:,} ({p['total'] / 1e6:.3f} M)")
print(f"  body            : {p['body']:,} ({p['body'] / 1e6:.3f} M)")
print(f"  embedding+head  : {p['embedding'] + p['head']:,}")
print(f"active body/token : {p['active_body_per_token']:,} "
      f"({p['active_body_per_token'] / 1e6:.3f} M, "
      f"{p['body_active_fraction']:.2%} of body)")
print("sinkhorn max err  :", p["sinkhorn_max_error"])
t = report["training"]
print(f"steps             : {t['steps']}  batch {t['global_batch']} x {t['seq_len']} tokens")
print(f"ms/step (mean)    : {t['mean_ms_per_step_after_first']:.1f}")
for k, v in report["eval"].items():
    print(f"eval reuse_p={k}: loss {v['loss']:.4f}  ce {v['ce']:.4f}  "
          f"acc {v['next_token_acc']:.3f}")
if report.get("generation"):
    g = report["generation"]
    print("generated         :", g["generated_ids"])
    print("greedy rule acc   :", g["greedy_rule_accuracy"])
print("first/last loss   :", t["history"][0]["loss"], "->", t["history"][-1]["loss"])

## 3. Notes

- Free Colab TPU = **1 device**, so `LOCED_EXPECTED_DEVICES=1` and the
  mesh gradient check reports `single device` (it needs ≥2 devices to
  compare a sharded gradient against a single-device one).
- The **sparse top-1 dispatch** is installed in the notebook process only;
  the repository is unchanged.